In [ ]:
# Cell 1 — Install everything
!pip install -q ollama flask pyngrok openai-whisper
# Replace TTS with edge-tts
!pip install -q edge-tts

# Install Ollama manually (Colab is Linux)
# Fix zstd first, then install ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.ai/install.sh | sh

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (13.1 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/lo

In [ ]:
from pyngrok import ngrok
import os

ngrok.kill()
os.system("fuser -k 5001/tcp")
os.system("fuser -k 5002/tcp")
print("Cleaned up")

Cleaned up


In [ ]:
import subprocess
import time

# Start Ollama server in background
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Ollama server started")

# Pull models (this takes 5-10 mins depending on connection)
print("Pulling llama3.1:8b ...")
!ollama pull llama3.1:8b

print("Pulling nomic-embed-text ...")
!ollama pull nomic-embed-text

print("Pulling deepseek-coder:6.7b ...")
!ollama pull deepseek-coder:6.7b

print("All models ready!")

Ollama server started
Pulling llama3.1:8b ...

Pulling nomic-embed-text ...

Pulling deepseek-coder:6.7b ...

All models ready!


In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": "Say hello in one sentence.",
        "stream": False
    }
)
print(response.json()["response"])

Hello, it's nice to meet you!


In [ ]:
from flask import Flask, request, jsonify, send_file, Response
import whisper, edge_tts, asyncio, tempfile, os, threading, time
import requests as req

app = Flask(__name__)
stt_model = whisper.load_model("small")
VOICE = "en-US-GuyNeural"
OLLAMA = "http://localhost:11434"

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

# Proxy ALL ollama routes transparently
@app.route("/api/generate", methods=["POST"])
def generate():
    r = req.post(f"{OLLAMA}/api/generate", json=request.json)
    return jsonify(r.json())

@app.route("/api/embeddings", methods=["POST"])
def embeddings():
    r = req.post(f"{OLLAMA}/api/embeddings", json=request.json)
    return jsonify(r.json())

@app.route("/api/tags", methods=["GET"])
def tags():
    r = req.get(f"{OLLAMA}/api/tags")
    return jsonify(r.json())

# REPLACE IT WITH:
@app.route("/transcribe", methods=["POST"])
def transcribe():
    audio = request.files["audio"]
    with tempfile.NamedTemporaryFile(delete=False, suffix=".webm") as f:
        audio.save(f.name)
        result = stt_model.transcribe(
            f.name,
            language="en",
            temperature=0.0,
            initial_prompt="Technical software engineering mock interview. Keywords: React, SQL, API, Git, database, loop, function, variable, docker, algorithm, system design, frontend, backend."
        )
        os.unlink(f.name)
    return jsonify({"text": result["text"]})


@app.route("/speak", methods=["POST"])
def speak():
    text = request.json.get("text", "")
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tmp.close()
    async def generate_audio():
        communicate = edge_tts.Communicate(text, VOICE)
        await communicate.save(tmp.name)
    asyncio.run(generate_audio())
    return send_file(tmp.name, mimetype="audio/mpeg")

threading.Thread(
    target=lambda: app.run(port=5001, use_reloader=False),
    daemon=True
).start()
time.sleep(3)
print("Server running on port 5001")

100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 289MiB/s]


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit


Server running on port 5001


In [ ]:

# Cell 4 — ngrok + Ollama tunnel
from pyngrok import ngrok, conf

NGROK_TOKEN = "3EaklfLrbyAoL5JzxcweF1qMYXp_675XAzuzZdTYRYGKWTfxE"
conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(5001, "http")
AI_SERVICE_URL = tunnel.public_url

print("\n========= COPY INTO YOUR LOCAL .env =========")
print(f"AI_SERVICE_URL={AI_SERVICE_URL}")
print("==============================================\n")


========= COPY INTO YOUR LOCAL .env =========
AI_SERVICE_URL=https://earwig-cinnamon-hatred.ngrok-free.dev



In [1]:
import time

print("Keep-alive active. Printing heartbeat every 20 minutes, up to 6 times (max 2 hours) to prevent idle timeouts.")

for i in range(6):
    # Wait for 20 minutes (20 mins * 60 seconds)
    time.sleep(20 * 60)

    elapsed_minutes = (i + 1) * 20
    print(f"Heartbeat - {elapsed_minutes} minutes elapsed ({i + 1}/6 times). Colab is running.")

print("Keep-alive session completed.")


Keep-alive active. Printing heartbeat every 20 minutes, up to 6 times (max 2 hours) to prevent idle timeouts.


KeyboardInterrupt: 